In [ ]:
import optuna
import random
import numpy as np
import torch
from utils.print_args import print_args
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast
import hiplot as hip
class Args:
    def __init__(self, dictionary):
        for key, value in dictionary.items():
            setattr(self, key, value)

def objective(trial):
    seq_len = trial.suggest_categorical('seq_len', [270, 540, 1080])
    label_len = trial.suggest_int('label_len', 10, 20)
    top_k = trial.suggest_int('top_k', 2, 10) 
    d_ff= trial.suggest_categorical('d_ff', [16, 64])
    d_model= trial.suggest_categorical('d_model', [8,16, 32, 64])
    
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    # 在 objective 函数内定义和使用 args_dict
    args_dict = { 
   'task_name': 'long_term_forecast',
    'is_training': 1,
    'model_id': 'tidal',
    'model': 'TimesNet',
    'data': 'custom',
    'root_path': './',
    'data_path': 'traffic.csv',
    'features': 'M',
    'target': 'Z1',
    'freq': 's',
    'checkpoints': './checkpoints/',
    'seq_len': seq_len,
    'label_len': label_len,
    'pred_len': 10,
    'seasonal_patterns': 'None',
    'inverse': False,
    'mask_rate': 0.25,
    'anomaly_ratio': 0.25,
    'top_k': top_k,
    'num_kernels': 6,
    'enc_in': 3,
    'dec_in': 3,
    'c_out': 3,
    'd_model': d_model,
    'n_heads': 8,
    'e_layers': 2,
    'd_layers': 1,
    'd_ff': d_ff,
    'moving_avg': 25,
    'factor': 1,
    'distil': True,
    'dropout': dropout,
    'embed': 'timeF',
    'activation': 'gelu',
    'output_attention': False,
    'channel_independence': 0,
    'num_workers': 10,
    'itr': 1,
    'train_epochs': 3,
    'batch_size': 128,
    'patience': 3,
    'learning_rate': learning_rate,
    'des': 'test',
    'loss': 'MSE',
    'lradj': 'type1',
    'use_amp': False,
    'use_gpu':True,
    'gpu': 0,
    'use_multi_gpu': False,
    'devices': '0,1,2,3',
    'p_hidden_dims': [128, 128],
    'p_hidden_layers': 2,} # 同之前的定义
   

    args = Args(args_dict)
    
    if args.use_gpu and args.use_multi_gpu:
        args.device_ids = [int(id_) for id_ in args.devices.replace(' ', '').split(',')]
        args.gpu = args.device_ids[0]
    torch.cuda.empty_cache()  # 释放显存
    # 进行实验
    Exp = Exp_Long_Term_Forecast(args)
    torch.cuda.empty_cache()  # 释放显存
    print('Args in experiment:')
    print_args(args)
    # 省略实际的训练逻辑，用 vali_loss 来模拟
    vali_loss = simulate_training(args) # 需要实现此函数
    return vali_loss
def simulate_training(args):
    Exp = Exp_Long_Term_Forecast
    vali_loss = None  # 初始化 vali_loss
    torch.cuda.empty_cache()  # 释放显存
    if args.is_training:
        for ii in range(args.itr):
            torch.cuda.empty_cache()  # 释放显存
            exp = Exp(args)  # Initialize the experiment with args
            # 构造 setting 字符串
            setting = construct_setting(args, ii)
            print(f'>>>>>>>start training : {setting}>>>>>>>>>>>>>>>>>>>>>>>>>>')
            _, vali_loss = exp.train(setting)  # 假设训练方法返回 None 和验证损失

    return vali_loss  # 确保在函数末尾返回验证损失

def construct_setting(args, ii):
    # 为了避免代码重复，提取构造 setting 字符串的逻辑到独立的函数
    setting = '{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
                args.task_name,
                args.model_id,
                args.model,
                args.data,
                args.features,
                args.seq_len,
                args.label_len,
                args.pred_len,
                args.d_model,
                args.n_heads,
                args.e_layers,
                args.d_layers,
                args.d_ff,
                args.factor,
                args.embed,
                args.distil,
                args.des, ii)
    return setting

if __name__ == '__main__':
    fix_seed = 2021
    random.seed(fix_seed)
    np.random.seed(fix_seed)
    torch.manual_seed(fix_seed)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=100, timeout=100000)
    
    print("Best trial:")
    trial = study.best_trial
    print("  Value: ", trial.value)
    print("  Params: ", trial.params)
hip.Experiment.from_optuna(study).display()